# 따릉이·버스·지하철 통합 경로 최적화

출발·도착 위치와 출발시각을 입력하여 도보, 따릉이, 버스, 지하철을 연결합니다. 따릉이는 V5.2.1 P50(중앙값)과 대여소 도착시점 재고 예측을 사용합니다. 대여·반납 시간을 따로 더하지 않으며 따릉이 이용 횟수에도 별도 제한을 두지 않습니다. 기존에 표시된 출력은 이전 실행 결과이므로 커널 재시작 후 Run All로 갱신하십시오.

입력을 수정하고 Run All을 누르십시오. 버스·지하철의 시간표와 좌표는 이미 받은 공식 파일을 읽습니다. 1~9호선 외에 경의중앙선·수인분당선·경춘선·공항철도·신분당선·우이신설선·신림선을 추가했습니다. 이 자료를 위한 서울시 API 호출이나 인증키는 필요하지 않습니다.

서울시 인증키는 현재 따릉이 재고 조회에만 사용합니다. 도보는 카카오, 날씨는 기상청 API를 사용합니다. 모델 재학습은 하지 않습니다. holidays가 없다면 최초 실행에서 현재 커널에 설치합니다.

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import os, sys, json, getpass, importlib.util
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p/'src'/'scheduled_runtime.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('bicycle 프로젝트 폴더에서 노트북을 여십시오.')
os.chdir(ROOT)
if str(ROOT/'src') not in sys.path: sys.path.insert(0, str(ROOT/'src'))
if importlib.util.find_spec('holidays') is None:
    import subprocess
    print('최초 실행: 공휴일 구분 라이브러리를 설치합니다.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'holidays>=0.60,<1'])
    importlib.invalidate_caches()
missing = [name for name in ['requests','holidays','openpyxl','bs4','pandas','numpy','torch','sklearn'] if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError('현재 커널에 필요한 라이브러리: '+', '.join(missing)+' / requirements.txt를 설치하고 커널을 다시 시작하세요.')
from project_paths import load_project_env, RUNTIME_OUTPUT_DIR
load_project_env()
KST = ZoneInfo('Asia/Seoul')
TRANSIT_DIR = ROOT/'data'/'transit'
OUTPUT_DIR = RUNTIME_OUTPUT_DIR/'scheduled'
print('프로젝트:', ROOT)


## 1. 검색 입력

DEPARTURE = None이면 자료 준비가 끝난 시점부터 5분 뒤에 출발합니다. 직접 지정할 때는 '2026-10-16T18:30:00+09:00'처럼 입력하십시오. 재고모델은 조회시점부터 60분 이내의 대여소 도착을 지원하며, 범위를 넘는 따릉이 탑승 후보는 제외됩니다.

총 경로시간은 대기·환승을 포함한 계획시간입니다. 따릉이 구간은 P50을 사용하며, 전체 여정의 도착확률이 50%라는 뜻은 아닙니다. 재고 위험 제한과 탐색 후보 제한은 그대로 유지합니다.

In [ ]:
ORIGIN = {'name':'서울역', 'latitude':37.5568, 'longitude':126.9723}
DESTINATION = {'name':'여의도 한강공원', 'latitude':37.5284, 'longitude':126.9326}
DEPARTURE = None
HORIZON_MINUTES = 180
SEARCH_MODE = 'integrated'  # integrated: 통합 / transit: 대중교통(버스·지하철) / bicycle: 따릉이
if SEARCH_MODE not in ('integrated','transit','subway','bicycle'):
    raise ValueError('SEARCH_MODE는 integrated/transit/bicycle입니다.')
USES_BIKE = SEARCH_MODE not in ('subway','transit')
OUTPUT_DIR = RUNTIME_OUTPUT_DIR/'scheduled'
if SEARCH_MODE != 'integrated': OUTPUT_DIR = OUTPUT_DIR/SEARCH_MODE
USE_KMA = True
USE_INVENTORY = True
USER_FACTOR = 1.0
PERSONALIZATION_ID = os.getenv('BICYCLE_PERSONALIZATION_ID') or None  # 선택, 기록이 없으면 배율 1.0
MAX_KAKAO_CALLS = 80
TOP_K = 3
SEARCH_ALGORITHM = 'astar'  # 비교용: 'dijkstra'

def query_departure():
    return DEPARTURE or (datetime.now(KST)+timedelta(minutes=5)).replace(microsecond=0).isoformat()

from multimodal import timestamp
timestamp(query_departure())
for name in ([] if SEARCH_MODE == 'bicycle' else ['bus_raw.zip','bus_route_nodes.dat']):
    if not (TRANSIT_DIR/name).is_file():
        raise FileNotFoundError('버스 원자료가 없습니다: '+str(TRANSIT_DIR/name))
print('자료 준비 완료 후 출발시각을 확정합니다.' if DEPARTURE is None else '출발: '+DEPARTURE)


## 2. 인증키

서울시 인증키는 USE_INVENTORY=True일 때 현재 따릉이 재고를 조회하는 용도입니다. 시간표·정류장·역 좌표는 모두 로컬 데이터 파일로 읽습니다. 카카오 REST API 키와 기상청 키는 각각 도보 경로·예보에 사용합니다. 키는 출력·저장하지 않습니다.

In [ ]:
if USES_BIKE and USE_INVENTORY and not os.getenv('SEOUL_API_KEY','').strip():
    from inventory_credentials import get_api_key
    stored = get_api_key()
    if stored: os.environ['SEOUL_API_KEY'] = stored
required = ['KAKAO_REST_API_KEY']
if USES_BIKE and USE_INVENTORY: required.append('SEOUL_API_KEY')
if USES_BIKE and USE_KMA and not (os.getenv('KMA_API_KEY') or os.getenv('KMA_AUTH_KEY')):
    required.append('KMA_API_KEY')
for key in required:
    if not os.getenv(key,'').strip():
        os.environ[key] = getpass.getpass(key+': ').strip()
    if not os.getenv(key,'').strip():
        raise ValueError(key+'가 비어 있습니다.')
print('인증키 준비 완료')


## 3. 다운로드된 시간표·좌표 연결

data/transit/files에 공식 원본 파일이 있습니다.

- 지하철 시간표: 2026년 9월 기준, 평일·토요일·공휴일 423,107행
- 버스 정류장 좌표: 2026년 9월 게시본
- 지하철 좌표: 서울교통공사 1~8호선·9호선 파일과 전국 도시철도 역사 파일

지하철은 열차번호·호선·방향·출발역·종착역으로 연결합니다. 버스는 요일별 첫차·막차·배차간격으로 기점 출발시각을 생성하고 전체 운행시간을 정류장 사이 거리 비율로 배분합니다.

운행시간·좌표가 없는 노선은 제외 건수를 집계합니다. 좌표를 연결하지 못한 역은 preparation_report.json에 남깁니다. 받은 시간표를 준비일로부터 14일 뒤 자동 만료시키지 않으며, 공식 파일이 갱신되면 원본을 교체해야 합니다. 추가 수도권 노선은 운영사 파일을 사용하며 공항철도의 2026-10-23 개정 전후를 구분합니다. 신분당선·우이신설선·신림선은 열차번호가 없는 분 단위 역별 출발시각이므로 열차 연결을 추정합니다. 종착역 도착시각 미제공 구간은 임의로 만들지 않습니다. 지도에는 OSM 도로·선로 형상을 사용하며 실내 환승이나 형상 미확보 구간은 개략도 점선으로 명확히 구분합니다. 도로 노드가 단절된 따릉이 후보는 제한 거리 안에서 연결 가능한 노드를 재탐색하며, 그래도 불가능하면 unroutable_bike_pairs에 기록합니다.

결과를 prepared.json에 저장하고 원본 파일이 같으면 재사용합니다. 대용량 시간표를 JSON으로 복제하지 않고 CSV에서 읽어 저장 용량을 줄입니다.

In [ ]:
print('따릉이 전용은 시간표를 준비하지 않습니다.' if SEARCH_MODE == 'bicycle' else
      '시간표는 모델 준비 후 확정된 출발시각에 맞춰 경로 탐색 단계에서 한 번 준비합니다.')


## 4. 검색시각 확정

현재 출발 모드는 여기서 출발시각을 다시 정합니다. 새 검색을 할 때는 이 셀부터 실행하십시오. 시간표를 구성하는 과정에는 API 호출이 없습니다.

In [ ]:
ACTUAL_DEPARTURE = DEPARTURE
print('모델 준비 후 출발시각·시간표·최신 재고를 확정합니다.' if DEPARTURE is None else '지정 출발: '+DEPARTURE)


## 5. 통합 경로 검색

기존 대여소·환승 연결을 유지하여 따릉이를 대중교통 전후·중간에 여러 번 이용할 수 있습니다. 대중교통 기본 경로 주변으로 탐색 범위를 제한하지 않습니다.

먼저 확보된 도보 연결과 시간표로 기본 경로를 찾습니다. 이후 도보 조회는 한 건씩, 따릉이 후보는 기본 두 개씩 처리하고 다음 지점의 탐색과 번갈아 진행합니다. 아직 확인하지 않은 후보는 삭제하지 않고 대기열에 보관합니다. 대여소 사이 도보 연결은 제외합니다.

같은 OD·5분 단위 출발시각·모델 입력 조건의 이동시간은 캐시를 재사용합니다. 탐색 순서만 변경하면 예측 캐시는 무효화하지 않으며, 실제 예측 코드·모델·날씨 조건이 바뀌면 구분합니다. 이번 통계 조회 최적화는 기존 피처 값과 정확히 같음을 검증한 빌드에 한해 기존 캐시와 호환됩니다. 재고는 실제 대여소 도착시각 기준으로 별도 예측합니다. 기존 모델·캐시 파일과 이전 출력은 유지되며 커널 재시작 후 Run All로 새 결과를 확인하십시오.

OD 통계는 약 100만 행 전체를 매번 결합하지 않고 준비된 인덱스로 조회합니다. 버스·지하철은 이전에 더 이른 도착·동등 이상 조건에서 이미 반영한 같은 운행편만 중복 처리를 생략합니다. 대여소 후보·거리 범위·다른 출발시각의 따릉이·재고 예측·중간 환승은 제거하지 않습니다.

P50 개인화는 선택 사항입니다. PERSONALIZATION_ID=None이면 기존 예측 그대로이며, 개인화 ID가 있어도 유효 기록이 없으면 배율 1.0입니다. 실제 대여·반납 기록이 쌓일 때만 기본 P50 대비 개인 배율을 갱신합니다. 가상 기록은 운영 데이터에 넣지 않습니다. 웹 연결·기록 API·현재 한계는 ../docs/PERSONALIZATION.md를 참고하십시오.

개인 이력 이상 판정의 기본 방식은 t 예측 상한입니다. 과거 log(실제시간/기본 P50)의 평균 + t값×표준편차×sqrt(1+1/n)을 사용합니다. PERSONALIZATION_HISTORY_METHOD를 비우거나 student_t로 두면 t 방식을 사용하며, rank는 비교용으로 명시할 수 있습니다. PERSONALIZATION_HISTORY_ALPHA의 기본값은 미설정이며 운영은 shadow 모드입니다. 설정해도 의심 표시만 남기고 자동 제외하지 않습니다. 표본이 2건 미만이거나 분산이 0이면 판정을 유보합니다. 가상 실험은 실제 개인 이력의 정규성·독립성이나 초과율을 검증한 결과가 아닙니다. 명시적인 운동·휴식 등의 기존 제외는 유지하며 경로 탐색의 기준은 P50으로 같습니다.

대중교통 정류장에서는 이후 운행편을 직접 살펴보므로 1분 단위 대기 상태를 반복 생성하지 않습니다. 재고 회복과 시간에 따른 이동시간 변화를 고려해야 하는 따릉이 대여소에서는 대기 탐색을 유지합니다.

기존 64회 예측 제한은 기본값에서 해제했습니다. 선택적으로 max_bike_predictions를 설정해도 새 AI 계산만 제한하고 도보·버스·지하철·캐시된 따릉이 구간 검색은 계속합니다. 전체 검색시간·상태 수 한도는 유지합니다. 한도 도달 시 partial로 표시하며 전역 최적을 보장하지 않습니다.

### 목적지 우선 탐색(A*)

기본값은 SEARCH_ALGORITHM='astar'입니다. 다만 현재처럼 도보 연결을 필요할 때 조회하는 교통망에서는 아직 모르는 도보 시간을 과대평가하지 않도록 하한을 0으로 둡니다. 도달한 상태의 우선순위는 Dijkstra와 같으며, 미완료 연결 작업은 별도 대기열에서 번갈아 처리합니다. 모든 연결이 알려진 고정 교통망에서만 완화 교통망의 A* 하한을 사용합니다.

기존 후보 연결과 시간별 상태는 유지합니다. 재고 부족 때문에 기다리거나 대중교통 사이에 따릉이를 여러 번 타는 경로도 탐색합니다. 하한을 구하지 못한 노드는 0으로 처리합니다. 총시간과 순위 점수에는 하한을 더하지 않습니다.

첫 경로를 찾았더라도 남은 후보가 더 빠를 가능성이 있으면 계속 확인합니다. 시간·조회·예측 한도에 걸리면 그때까지 찾은 경로를 partial로 반환합니다. 이 변경만으로 혼합 경로가 반드시 선택되거나 전역 최적이 보장되는 것은 아닙니다. 실제 속도와 추천 순위는 재실행으로 확인해야 합니다.

In [ ]:
from scheduled_runtime import ScheduledConfig, run_scheduled_optimization
config = ScheduledConfig(search_mode=SEARCH_MODE, horizon_minutes=HORIZON_MINUTES, top_k=TOP_K, bike_quantile='p50',
    max_bike_predictions=None, search_algorithm=SEARCH_ALGORITHM,
    bike_expansion_batch_size=2,  # 한 차례에 계산할 따릉이 후보 수
    reuse_transit_suffixes=True,  # 이미 반영한 동일 운행편 중복 처리 재사용
    max_seconds=30, complete_candidates=True, max_kakao_calls=MAX_KAKAO_CALLS,
    require_known_inventory=USE_INVENTORY and USES_BIKE,
    max_empty_probability=0.10 if USE_INVENTORY and USES_BIKE else None)
result = run_scheduled_optimization(ORIGIN, DESTINATION, ACTUAL_DEPARTURE,
    config=config, use_kma=USE_KMA and USES_BIKE,
    personalization_id=PERSONALIZATION_ID,
    use_inventory=USE_INVENTORY and USES_BIKE, user_factor=USER_FACTOR, output_dir=OUTPUT_DIR)
ACTUAL_DEPARTURE = result['departure']
print('출발:', ACTUAL_DEPARTURE)
display(result['network_metadata'].get('bus_report',{}))
display(result['network_metadata'].get('subway_report',{}))
print('상태:', result['status'], '/ 발견 경로:', len(result['routes']))
if result.get('mixed_routes_requested'):
    print('따릉이·대중교통 혼합 경로:', len(result.get('mixed_routes', [])), '개 (전체 순위와 별도 보존)')
print('종료 사유:', result.get('termination_reason') or '탐색 종료')
print('카카오 호출:', result['api_usage'])
print('저장 위치:', OUTPUT_DIR)

print('새 이동시간 예측 요청:', result['bike_predictions'])
print('이동시간·재고 재사용:', result.get('bicycle_checkpoint'))
print('예측 한도로 건너뛴 요청:', result.get('skipped_bike_requests',0))
print('재고 지원범위를 지난 대기 생략:', result.get('expired_bicycle_waits', 0))
print('확인된 막다른 후보의 예측 생략:', result.get('skipped_dead_end_predictions', 0))

print('탐색 방식:', result['search_algorithm'])
print('시간표 색인·하한 준비(초):', round(result['search_preparation_seconds'],3))
print('하한 계산(초):', round(result['heuristic_build_seconds'],3))
print('확장 상태 수:', result['expanded_states'])
print('연결 처리 방식:', result.get('expansion_policy'))
print('번갈아 처리한 묶음 수:', result.get('deferred_work_batches',0))
print('남은 연결 작업:', result.get('pending_work_items',0))
print('중복 반영을 재사용한 승차 상태:', result.get('reused_transit_expansions',0))
print('실제로 확인한 시간표 구간:', result.get('transit_edges_examined',0))
if result.get('transit_roundtrip_pruning'):
    print('불필요 왕복 차단(승차·하차 전환):', result.get('pruned_transit_roundtrips', 0))
personal = result.get('personalization', {})
print('개인화:', '사용' if personal.get('enabled') else '미사용', '/ 유효 기록:', personal.get('usable_records',0), '/ 배율:', round(personal.get('factor',1.0),4))


## 6. 결과 표·지도

같은 버스·열차를 계속 타는 구간은 하나로 묶어 표시합니다. 지도에서 버스·지하철은 정류장·역 좌표를 순서대로 연결하고, 따릉이와 도보는 확보한 경로를 표시합니다.

In [ ]:
import pandas as pd
from scheduled_report import route_summary, route_segments
display(pd.DataFrame(route_summary(result)))
for index, route in enumerate(result['routes']):
    print(f'경로 {index+1}: {route["total_minutes"]:.1f}분')
    display(pd.DataFrame(route_segments(result,index)).drop(columns=['운행편'],errors='ignore'))
if not result['routes']:
    print('현재 후보와 설정에서는 경로를 찾지 못했습니다. 시간표·도보 연결·재고 지원 범위를 확인하세요.')
if result.get('failure_examples'): display(pd.DataFrame(result['failure_examples']))
print('도보 연결 한도 도달:', result['network_metadata'].get('connector_limit_reached',False))


In [ ]:
from IPython.display import HTML, display
import html
map_document = (OUTPUT_DIR/'routes.html').read_text(encoding='utf-8')
display(HTML('<iframe style="width:100%;height:650px;border:0" srcdoc="'+html.escape(map_document,quote=True)+'"></iframe>'))
print('지도 파일:', OUTPUT_DIR/'routes.html')


## 저장 결과와 웹 연결

- timetable.json: 검색시각에 맞춘 대중교통 운행편
- network.json: 도보·따릉이 후보를 연결한 교통망
- result.json: 추천 경로, 구간별 시각, 예측시간, 재고 판정
- routes.html: 경로별 지도

웹에서는 scheduled_runtime.run_scheduled_optimization()에 같은 입력을 전달합니다. 시간표·좌표는 data/transit/prepared.json과 files의 원본 CSV를 읽습니다. 원본 파일의 출처와 해시는 data/transit/files/sources.json에 있습니다.

자료 출처: [버스 정류장 좌표](https://data.seoul.go.kr/dataList/OA-15067/F/1/datasetView.do), [지하철 시간표](https://www.data.go.kr/data/15098251/fileData.do), [1~8호선 좌표](https://www.data.go.kr/data/15099316/fileData.do), [9호선 좌표](https://www.data.go.kr/data/15041335/fileData.do), [전국 도시철도 역사](https://data.kric.go.kr/rips/M_01_01/detail.do?id=32).